In [ ]:
import joblib
import optuna
from optuna.samplers import RandomSampler, TPESampler
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances, plot_slice, plot_parallel_coordinate
import numpy as np
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.linear_model import SGDClassifier, SGDRegressor, LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor 
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score
from fastparquet import write
from fastparquet import ParquetFile
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-08-16-04_27_51_PM'

In [ ]:
oof_experiment_config = {
     "experiment": {
        "model": "xgboost",
        "type": "optuna",
        "num_trials": 10,
        "train_feature_groups": ["baseline/train.parq", "features/train.parq", "target_encode/train.parq"],
        "test_feature_groups": ["baseline/test.parq", "features/test.parq", "target_encode/test.parq"],
        "description": "xgb + clf + baseline + features + TE + optuna tune",
        "target": "classification",
    },

    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 0
    },

    "features": {
        "class_sample_weight": False
    },

    "params": {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 5000,
        "enable_categorical": True,
        "early_stopping_rounds": 50,
        "device": "cuda",
        "n_jobs": -1,
        "verbosity": 0
    },
    
    "fit_params": {
    }
}
oof_experiment_config["experiment"]["id"] = f"{dt_str}_{oof_experiment_config["experiment"]["model"]}"
experiment_config = oof_experiment_config.copy()

In [ ]:
def tune_params(trial):
    tree_method = trial.suggest_categorical("tree_method", ["hist", "approx"])
    booster = trial.suggest_categorical("booster", ["gbtree"])
    grow_policy = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])

    if grow_policy == "lossguide":
        max_depth = 0
        max_leaves = trial.suggest_int('max_leaves', 8, 512)
    else:
        max_depth = trial.suggest_int('max_depth', 2, 15)
        max_leaves = 0

    trial_params = {
        'max_depth': max_depth,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.4, log=True),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        "booster": booster,
        "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_float('min_child_weight', 1, 20, log=True),
        "max_bin": trial.suggest_int('max_bin', 32, 256, step=10),
        "tree_method": tree_method,
        "grow_policy": grow_policy
    }

    if grow_policy == "lossguide":
        trial_params["max_leaves"] = max_leaves

    return trial_params 

In [ ]:
sampler = RandomSampler()

In [4]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

In [5]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [ ]:
raw_train_df = pd.read_csv(f"{data_path}/raw/train.csv")
raw_train_id = raw_train_df["id"]
X_dfs, X_test_dfs = [], []

for fg in oof_experiment_config["experiment"]["train_feature_groups"]:
    file_path = f"{data_path}/{fg}"
    df = ParquetFile(file_path).to_pandas()
    X_dfs.append(df)

for fg in oof_experiment_config["experiment"]["test_feature_groups"]:
    file_path = f"{data_path}/{fg}"
    df = ParquetFile(file_path).to_pandas()
    X_test_dfs.append(df)

X = pd.concat(X_dfs, axis=1)
X_test = pd.concat(X_test_dfs, axis=1)
y = raw_train_df[target_column]

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 40 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   age                               662440 non-null  float64 
 1   daily_screen_time_hours           595515 non-null  float64 
 2   social_media_hours                557374 non-null  float64 
 3   gaming_hours                      564548 non-null  float64 
 4   work_study_hours                  639851 non-null  float64 
 5   sleep_hours                       646889 non-null  float64 
 6   notifications_per_day             623785 non-null  float64 
 7   app_opens_per_day                 610659 non-null  float64 
 8   weekend_screen_time               579306 non-null  float64 
 9   gender                            662335 non-null  category
 10  stress_level                      636221 non-null  float64 
 11  academic_work_impact              647145 non-null 

In [ ]:
def make_model(config, optuna_params):
    name = config["experiment"]["model"]
    target = config["experiment"]["target"]
    params = config["params"]

    model_dict = {
        "adaboost": (AdaBoostClassifier, AdaBoostRegressor),
        "gradientboost": (GradientBoostingClassifier, GradientBoostingRegressor),
        "catboost": (CatBoostClassifier, CatBoostRegressor), # TODO - fix missing col issue in catboost
        "xgboost": (XGBClassifier, XGBRegressor),
        "lightgbm": (LGBMClassifier, LGBMRegressor),
        "randomforest": (RandomForestClassifier, RandomForestRegressor),
        "extratrees": (ExtraTreesClassifier, ExtraTreesRegressor),
        "hgbc": (HistGradientBoostingClassifier, HistGradientBoostingRegressor),
        "knn": (KNeighborsClassifier, KNeighborsRegressor),
        "sgd": (SGDClassifier, SGDRegressor),
        "linear": (LogisticRegression, LinearRegression),
        "decisiontree": (DecisionTreeClassifier, DecisionTreeRegressor),
        "mlp": (MLPClassifier, MLPRegressor),
        "tabpfn": (TabPFNClassifier, TabPFNRegressor),
    }

    target_index = target == "regression"
    model_class = model_dict[name][target_index]

    return model_class(**params, **optuna_params)

In [ ]:
def oof_fit(config, model, X_train, y_train, X_valid, y_valid):
    name = config["experiment"]["model"]
    no_eval_models = ["adaboost", "gradientboost", "randomforest", "extratrees", "knn", "sgd", "linear", "decisiontree", "mlp", "tabpfn"]
    fit_params = config["fit_params"]
    
    if "features" in config and "class_sample_weight" in config["features"] and config["features"]["class_sample_weight"]:
        train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
    else:
        train_sample_weight = None

    if name in no_eval_models:
        model.fit(X_train, y_train, sample_weight=train_sample_weight)
    elif name == "hgbc":
        model.fit(X_train, y_train, X_val=X_valid, y_val=y_valid, sample_weight=train_sample_weight)
    elif name == "lightgbm":
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], sample_weight=train_sample_weight, **fit_params)
    else:
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], sample_weight=train_sample_weight, verbose=False)

In [10]:
def predict(config, model, X_feat):
    task = config["experiment"]["target"]
    if task == "classification":
        return model.predict_proba(X_feat)[:, 1]
    else:
        return model.predict(X_feat)

In [ ]:
cv_config = experiment_config["cv"]
kf = StratifiedKFold(n_splits=cv_config["n_splits"], random_state=cv_config["random_state"], shuffle=cv_config["shuffle"])
y_cv = pd.Series(index=y.index, dtype=float, name=target_column)

train_index, valid_index = next(kf.split(X, y))

X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

def objective(trial):
    trial_params = tune_params(trial)
    model = make_model(oof_experiment_config, trial_params)

    oof_fit(
        oof_experiment_config,
        model,
        X_train,
        y_train,
        X_valid,
        y_valid
    )

    y_pred = predict(oof_experiment_config, model, X_valid)

    return roc_auc_score(y_valid, y_pred)

In [ ]:
study = optuna.create_study(sampler=sampler, direction='maximize')
study.optimize(objective, n_trials=oof_experiment_config['experiment']['num_trials'])

In [ ]:
df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
df

In [ ]:
study.best_trial.params

In [ ]:
plot_optimization_history(study)

In [ ]:
plot_param_importances(study)

In [ ]:
plot_parallel_coordinate(study)

In [ ]:
plot_slice(study)